In [ ]:
import kagglehub

path = kagglehub.dataset_download("ahemateja19bec1025/traffic-sign-dataset-classification")

print("Path to dataset files:", path)

In [ ]:
import os
import tensorflow as tf

base_data_dir = os.path.join(path, 'traffic_Data')

train_dir = os.path.join(base_data_dir, 'DATA')
test_dir = os.path.join(base_data_dir, 'TEST')

print(f"Training data directory: {train_dir}")
print(f"Test data directory: {test_dir}")

if not os.path.exists(train_dir):
    print(f"Warning: Training directory not found at {train_dir}")
if not os.path.exists(test_dir):
    print(f"Warning: Test directory not found at {test_dir}")

In [ ]:
tf.random.set_seed(42)

img_height = 128
img_width = 128

all_class_dirs_str = sorted([d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))], key=int)

class_name_to_true_label_map = {name: int(name) for name in all_class_dirs_str}

global num_classes
num_classes = len(all_class_dirs_str)
print(f"Corrected number of classes: {num_classes}")

global class_names
class_names = all_class_dirs_str
print(f"Corrected class names: {class_names}")

# Collect train image paths and their true labels
train_image_paths_list = []
train_labels_list = []
for class_name_str in all_class_dirs_str:
    class_folder = os.path.join(train_dir, class_name_str)
    true_label = class_name_to_true_label_map[class_name_str]
    for fname in os.listdir(class_folder):
        if fname.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
            train_image_paths_list.append(os.path.join(class_folder, fname))
            train_labels_list.append(true_label)

# Collect test image paths and their true labels
test_image_paths = sorted([
    os.path.join(test_dir, fname) for fname in os.listdir(test_dir)
    if fname.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif'))
])
test_labels_list = []
for img_path in test_image_paths:
    filename = os.path.basename(img_path)
    try:
        class_id_str = filename.split('_')[0]
        true_label = class_name_to_true_label_map[str(int(class_id_str))]
        test_labels_list.append(true_label)
    except (ValueError, IndexError, KeyError) as e:
        print(f"Could not infer or map class ID from filename: {filename}. Skipping. Error: {e}")

# Define a function to load and preprocess images
def preprocess_image(image_path, label):
    img = tf.io.read_file(image_path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [img_height, img_width])
    img = tf.cast(img, tf.float32) / 255.0
    return img, label

train_ds_raw = tf.data.Dataset.from_tensor_slices((train_image_paths_list, train_labels_list))
train_ds = train_ds_raw.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

test_ds_raw = tf.data.Dataset.from_tensor_slices((test_image_paths, test_labels_list))
test_ds = test_ds_raw.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

print(f"Number of elements in training set: {tf.data.experimental.cardinality(train_ds).numpy()}")
print(f"Number of elements in testing set: {tf.data.experimental.cardinality(test_ds).numpy()}")

### 1. Prepare Data for PyTorch
First, we need to convert the `tf.data.Dataset` objects (`train_ds` and `test_ds`) into PyTorch tensors and then create `DataLoader`s. Images will be transposed from `(height, width, channels)` to `(channels, height, width)` as expected by PyTorch's convolutional layers.

In [ ]:
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

# Extract images and labels from train_ds
train_images = []
train_labels = []
for img, label in train_ds:
    train_images.append(img.numpy())
    train_labels.append(label.numpy())

train_images = np.array(train_images)
train_labels = np.array(train_labels)

# Extract images and labels from test_ds
test_images = []
test_labels_np = []
for img, label in test_ds:
    test_images.append(img.numpy())
    test_labels_np.append(label.numpy())

test_images = np.array(test_images)
test_labels_np = np.array(test_labels_np)

print(f"Shape of train_images: {train_images.shape}")
print(f"Shape of train_labels: {train_labels.shape}")
print(f"Shape of test_images: {test_images.shape}")
print(f"Shape of test_labels_np: {test_labels_np.shape}")

# Convert numpy arrays to torch tensors
train_images_torch = torch.from_numpy(train_images).permute(0, 3, 1, 2).float()
train_labels_torch = torch.from_numpy(train_labels).long()

test_images_torch = torch.from_numpy(test_images).permute(0, 3, 1, 2).float()
test_labels_torch = torch.from_numpy(test_labels_np).long()

# Create TensorDatasets
train_dataset = TensorDataset(train_images_torch, train_labels_torch)
test_dataset = TensorDataset(test_images_torch, test_labels_torch)

# Create DataLoaders
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"\nNumber of batches in train_loader: {len(train_loader)}")
print(f"Number of batches in test_loader: {len(test_loader)}")

### 2. Define the CNN Model using `torch.nn`
This model is a simple Convolutional Neural Network (CNN) similar to the Keras model that was previously deleted, adapted for PyTorch. It consists of three convolutional layers, followed by max-pooling layers (after the first two conv layers), and then fully connected layers for classification.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt


# Define the CNN model
num_classes = len(class_names)

# class SimpleCNN(nn.Module):
#     def __init__(self, num_classes):
#         super().__init__()
#         self.features = nn.Sequential(
#             nn.Conv2d(3, 32, kernel_size=3, padding=1),
#             nn.ReLU(inplace=True),
#             nn.MaxPool2d(2),  # 128x128 -> 64x64

#             nn.Conv2d(32, 64, kernel_size=3, padding=1),
#             nn.ReLU(inplace=True),
#             nn.MaxPool2d(2),  # 64x64 -> 32x32

#             nn.Conv2d(64, 128, kernel_size=3, padding=1),
#             nn.ReLU(inplace=True),
#         )

#         # After features, tensor shape will be (Batch, 128, 32, 32)
#         # Flattened size: 128 * 32 * 32 = 131072
#         self.classifier = nn.Sequential(
#             nn.Flatten(),
#             nn.Linear(128 * 32 * 32, 256), # Adjusted for 128x128 input
#             nn.ReLU(inplace=True),
#             nn.Dropout(0.3),
#             nn.Linear(256, num_classes),
#         )

#     def forward(self, x):
#         x = self.features(x)
#         x = self.classifier(x)
#         return x



class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        self.dropout = nn.Dropout(0.5)

        self.fc1 = nn.Linear(64 * 32 * 32, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = F.relu(self.bn3(self.conv3(x)))

        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

model = SimpleCNN(num_classes)
print(model)


### 3. Define Loss Function and Optimizer
We'll use `CrossEntropyLoss` for classification and the `Adam` optimizer.

In [ ]:
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

print(f"Model will be trained on: {device}")

### 4. Train the Model
Now, let's train the `SimpleCNN` model using the `train_loader` and evaluate its performance on the `test_loader`.

In [ ]:
num_epochs = 20

train_images_tensor = train_images_torch
train_labels_tensor = train_labels_torch

num_train_samples = len(train_images_tensor)
batch_size = BATCH_SIZE # 32

print(f"Starting training with Learning Rate: {optimizer.param_groups[0]['lr']}")

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    indices = torch.randperm(num_train_samples)

    for i in range(0, num_train_samples, batch_size):
        batch_indices = indices[i:i + batch_size]
        inputs = train_images_tensor[batch_indices].to(device)
        labels = train_labels_tensor[batch_indices].to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    train_accuracy = 100 * correct_train / total_train
    avg_train_loss = running_loss / (num_train_samples / batch_size) if (num_train_samples / batch_size) > 0 else 0
    print(f'Epoch {epoch + 1}, Loss: {avg_train_loss:.4f}, Train Accuracy: {train_accuracy:.2f}%')

    # Evaluate on the test set
    model.eval()
    correct_test = 0
    total_test = 0
    with torch.no_grad():
        for data in test_loader:
            images, labels = data[0].to(device), data[1].to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total_test += labels.size(0)
            correct_test += (predicted == labels).sum().item()

    test_accuracy = 100 * correct_test / total_test
    print(f'Test Accuracy after Epoch {epoch + 1}: {test_accuracy:.2f}%')

print('\nFinished Training')

In [ ]:
sample_idx = 1111
sample_traffic_sign = test_images_torch[sample_idx].unsqueeze(0).to(device)

img_to_show = sample_traffic_sign.squeeze(0).cpu().numpy()
img_to_show = img_to_show.transpose(1, 2, 0)

plt.figure(figsize=(4, 4))
plt.imshow(img_to_show)
plt.title(f"Sample Traffic Sign (Class {test_labels_torch[sample_idx].item()})")
plt.axis('off')
plt.show()

In [ ]:
model.eval()
with torch.no_grad():
    outputs = model(sample_traffic_sign)
    probs = torch.softmax(outputs, dim=1)

    # Get the predicted class and its confidence
    # Depending on the RNG used in training the model, this iteration of the model may or may not even guess the correct class in the first training attempt
    pred_class_idx = probs.argmax(dim=1).item()
    confidence = probs[0, pred_class_idx].item()

print(f"Predicted Class: {class_names[pred_class_idx]} (Index: {pred_class_idx})")
print(f"Confidence: {confidence:.4f}")

In [ ]:
def fgsm_attack(model, x, y, epsilon=0.03):
    """
    x is normalized input (expected in [0, 1] range for this model).
    Returns:
        x_adv: adversarial example in normalized space
        grad: gradient wrt input
    """
    model.eval()

    x_adv = x.clone().detach().requires_grad_(True)
    logits = model(x_adv)
    loss = criterion(logits, y)

    model.zero_grad()
    loss.backward()

    grad = x_adv.grad.detach()
    x_adv = x_adv + epsilon * grad.sign()

    x_adv = torch.clamp(x_adv, 0, 1)

    return x_adv.detach(), grad

In [ ]:
# Get the true label for the sample
true_label_for_sample = test_labels_torch[sample_idx].unsqueeze(0).to(device)

# Create the adversarial traffic sign
epsilon = 0.03
adversarial_traffic_sign, grad0 = fgsm_attack(model, sample_traffic_sign, true_label_for_sample, epsilon=epsilon)

In [ ]:
model.eval()
with torch.no_grad():
    outputs_adv = model(adversarial_traffic_sign)
    probs_adv = torch.softmax(outputs_adv, dim=1)

    # Get the predicted class and its confidence for the adversarial example
    pred_adv_class_idx = probs_adv.argmax(dim=1).item()
    confidence_adv = probs_adv[0, pred_adv_class_idx].item()

print(f"Adversarial Predicted Class: {class_names[pred_adv_class_idx]} (Index: {pred_adv_class_idx})")
print(f"Adversarial Confidence: {confidence_adv:.4f}")

In [ ]:
img_to_show_adv = adversarial_traffic_sign.squeeze(0).cpu().numpy()
img_to_show_adv = img_to_show_adv.transpose(1, 2, 0)

plt.figure(figsize=(4, 4))
plt.imshow(img_to_show_adv)
plt.title(f"Adversarial Sample\nTrue Class: {class_names[true_label_for_sample.item()]}\nPrediction: {class_names[pred_adv_class_idx]}\nConfidence: {confidence_adv:.3f}")
plt.axis('off')
plt.show()

In [ ]:
grad0_np = grad0.squeeze(0).cpu().numpy()
grad0_np = np.transpose(grad0_np, (1, 2, 0))

plt.figure(figsize=(4, 4))
max_grad_val = np.max(np.abs(grad0_np[:, :, 0]))
plt.imshow(grad0_np[:, :, 0], cmap='RdBu', vmin=-max_grad_val, vmax=max_grad_val)
plt.colorbar(label='Signed Gradient Magnitude')
plt.title("Signed Gradient Heatmap")
plt.axis('off')
plt.show()